# KIAS Maze Multi-Agent Experiment

이 노트북은 Maze path finding에서 agent 수, maze 난이도, 합의 방식에 따른 accuracy와 cost를 비교하기 위한 실행 파일입니다.

중요한 점: agent 수가 늘어나도 Qwen 모델은 한 번만 로드합니다. agent는 서로 다른 prompt와 응답 기록으로 시뮬레이션합니다.

## 1. 패키지 설치

서버에서 처음 한 번만 실행하세요. CUDA/PyTorch가 이미 설치되어 있다면 두 번째 줄만 실행하는 편이 안전할 수 있습니다.

In [ ]:
# 처음 한 번만 실행
# %pip install -U -r requirements.txt

# PyTorch/CUDA가 이미 서버에 설치되어 있으면 이쪽을 먼저 권장
# %pip install -U transformers accelerate pandas matplotlib bitsandbytes

## 2. 코드 불러오기

In [ ]:
from pathlib import Path

from maze_agents_experiment import (
    MockLLM,
    LocalHFLLM,
    create_maze_case,
    run_single_agent,
    run_majority_vote,
    run_experiment_grid,
    summarize_results_csv,
    plot_accuracy_cost,
    evaluate_candidate,
    save_maze_image,
)

Path('results').mkdir(exist_ok=True)

## 3. MockLLM으로 코드 동작 확인

이 단계는 Qwen을 부르지 않습니다. 전체 코드가 정상적으로 돌아가는지만 확인합니다.

In [ ]:
mock = MockLLM()
maze = create_maze_case(5, 5, seed=1, difficulty_name='easy')

print('Start:', maze.start)
print('Goal:', maze.goal)
print('Shortest path length:', len(maze.solution_path) - 1)

mock_result = run_majority_vote(maze, mock, n_agents=3)
print('Mock cost:', mock_result.cost)
print('Mock evaluation:', evaluate_candidate(maze, mock_result.final_candidate))

save_maze_image(maze, 'results/demo_easy_maze_solution.png', show_solution=True)
print('Saved maze image: results/demo_easy_maze_solution.png')

## 4. Qwen 모델 로드

처음에는 `Qwen/Qwen2.5-1.5B-Instruct`를 추천합니다. GPU 메모리가 부족하면 `load_in_4bit=True`로 바꾸세요.

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

llm = LocalHFLLM(
    model_name=MODEL_NAME,
    max_new_tokens=512,
    temperature=0.2,
    top_p=0.9,
    load_in_4bit=False,  # 메모리가 부족하면 True로 변경
)

llm.load()
print('Loaded:', MODEL_NAME)

## 5. Qwen single-agent quick test

전체 실험 전에 easy maze 하나만 풀려 봅니다.

In [ ]:
maze = create_maze_case(5, 5, seed=1, difficulty_name='easy')
single_result = run_single_agent(maze, llm)

print('Raw model answer:')
print(single_result.answers[0].raw_text)
print('\nEvaluation:')
print(evaluate_candidate(maze, single_result.final_candidate))
print('\nCost:')
print(single_result.cost)

## 6. 전체 실험 실행

처음에는 작은 설정으로 돌리고, 서버가 안정적이면 `hard` 크기, `SEEDS`, `AGENT_COUNTS`를 늘리세요.

`agent_count=1`은 consensus method별로 반복하지 않고 single-agent baseline으로 한 번만 기록됩니다.

In [ ]:
DIFFICULTIES = {
    'easy': {'sizex': 5, 'sizey': 5},
    'medium': {'sizex': 8, 'sizey': 8},
    'hard': {'sizex': 10, 'sizey': 10},
}

SEEDS = [1, 2, 3]
AGENT_COUNTS = [1, 3, 5]  # 서버가 안정적이면 [1, 3, 5, 10]
METHODS = ['majority_vote', 'debate_vote', 'judge_consensus']

csv_path = run_experiment_grid(
    llm=llm,
    difficulties=DIFFICULTIES,
    seeds=SEEDS,
    agent_counts=AGENT_COUNTS,
    methods=METHODS,
    output_dir='results',
    debate_rounds=1,
    save_mazes=True,
)

print('Saved:', csv_path)

## 7. 결과 요약과 그래프

In [ ]:
summary = summarize_results_csv(csv_path, output_path='results/summary.csv')
display(summary)

plot_accuracy_cost(csv_path, output_dir='results')
print('Saved summary: results/summary.csv')
print('Saved plots under results/')

## 8. 원본 결과 확인

In [ ]:
import pandas as pd

df = pd.read_csv(csv_path)
display(df.head())
display(df[['difficulty', 'method', 'agent_count', 'success', 'valid_steps', 'optimal', 'total_tokens', 'generation_time_sec']])